In [1]:
# check lodhi road data for aug

import pandas as pd

FILE = r"D:\miniproject\data\bronze\delhi_air_weather.csv"

df = pd.read_csv(FILE)

# read values for lodhi road station for the month of august
lodhi_aug = df[(df["station_name"] == "Lodhi Road") & (df["timestamp"].str.startswith("2025-08"))]

print(lodhi_aug)

                timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  \
5088  2025-08-01 00:00:00            NaN           NaN         NaN   
5089  2025-08-01 01:00:00            NaN           NaN         NaN   
5090  2025-08-01 02:00:00            NaN           NaN         NaN   
5091  2025-08-01 03:00:00            NaN           NaN         NaN   
5092  2025-08-01 04:00:00            NaN           NaN         NaN   
...                   ...            ...           ...         ...   
5827  2025-08-31 19:00:00            NaN           NaN         NaN   
5828  2025-08-31 20:00:00            NaN           NaN         NaN   
5829  2025-08-31 21:00:00            NaN           NaN         NaN   
5830  2025-08-31 22:00:00            NaN           NaN         NaN   
5831  2025-08-31 23:00:00            NaN           NaN         NaN   

      NO2 (µg/m³)  NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  \
5088          NaN        NaN          NaN          NaN         NaN   
5089          NaN  

In [2]:
# fill_lodhi_pm25_from_shadipur_aug2025.py
import pandas as pd

FILE = r"D:\miniproject\data\bronze\delhi_air_weather.csv"
OUT  = r"D:\miniproject\data\bronze\delhi_air_weather_filled.csv"

PM  = "PM2.5 (µg/m³)"
SRC = "Shadipur"      # station 3 — copy FROM
DST = "Lodhi Road"    # station 2 — fill INTO

df = pd.read_csv(FILE)

# --- August 2025 mask (year-specific; add more years if needed) ---
aug = df["timestamp"].str.startswith("2025-08")

# --- Diagnostics before fill ---
aug_dst = df.loc[aug & (df["station_name"] == DST)]
aug_src = df.loc[aug & (df["station_name"] == SRC)]
print(f"{DST:12s} Aug rows: {len(aug_dst):>5}  "
      f"PM2.5 non-null: {aug_dst[PM].notna().sum():>5}  "
      f"null: {aug_dst[PM].isna().sum():>5}")
print(f"{SRC:12s} Aug rows: {len(aug_src):>5}  "
      f"PM2.5 non-null: {aug_src[PM].notna().sum():>5}  "
      f"null: {aug_src[PM].isna().sum():>5}")

# --- Source lookup: timestamp -> Shadipur PM2.5 (non-null only) ---
src_map = (
    aug_src.loc[aug_src[PM].notna(), ["timestamp", PM]]
           .drop_duplicates("timestamp")
           .set_index("timestamp")[PM]
)

# --- Rows to fill: Lodhi Road, August, PM2.5 is null ---
dst_mask = aug & (df["station_name"] == DST) & df[PM].isna()

# --- Narrow to rows where Shadipur actually has a value at that timestamp ---
src_vals  = df.loc[dst_mask, "timestamp"].map(src_map)
fill_mask = dst_mask & src_vals.notna()

# --- Apply the fill ---
df.loc[fill_mask, PM] = src_vals[fill_mask]

# --- Audit column so you can exclude imputed rows later ---
df["pm25_filled_from_shadipur"] = False
df.loc[fill_mask, "pm25_filled_from_shadipur"] = True

# --- Diagnostics after fill ---
aug_dst_after = df.loc[aug & (df["station_name"] == DST)]
print(f"\nFilled: {int(fill_mask.sum())} rows "
      f"(out of {int(dst_mask.sum())} null rows that matched a timestamp)")
print(f"{DST} Aug PM2.5 non-null after: {aug_dst_after[PM].notna().sum()}")
print(f"{DST} Aug PM2.5 null     after: {aug_dst_after[PM].isna().sum()}")

# --- Save ---
#df.to_csv(OUT, index=False)
print(f"\n To be Saved to -> {OUT}")

Lodhi Road   Aug rows:   744  PM2.5 non-null:     0  null:   744
Shadipur     Aug rows:   744  PM2.5 non-null:   727  null:    17

Filled: 727 rows (out of 744 null rows that matched a timestamp)
Lodhi Road Aug PM2.5 non-null after: 727
Lodhi Road Aug PM2.5 null     after: 17

 To be Saved to -> D:\miniproject\data\bronze\delhi_air_weather_filled.csv


In [ ]:
# null_analysis.py
# import pandas as pd

# FILE = r"D:\miniproject\data\bronze\delhi_air_weather.csv"

POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)',
              'NH3 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
METEO = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
         'wind_direction_10m', 'rain', 'surface_pressure',
         'boundary_layer_height', 'wind_u', 'wind_v']
COLS = POLLUTANTS + METEO

# --- sanity: columns present? ---
missing = [c for c in COLS if c not in df.columns]
if missing:
    print("!! Missing columns in file:", missing)
    COLS = [c for c in COLS if c in df.columns]

# --- scope ---
aug = df["timestamp"].str.startswith("2025-08")
stations = df["station_name"].dropna().unique().tolist()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

def null_table(frame, cols, label):
    n = len(frame)
    tbl = pd.DataFrame({
        "non_null": frame[cols].notna().sum(),
        "null":     frame[cols].isna().sum(),
    })
    tbl["null_%"]  = (tbl["null"] / n * 100).round(1) if n else 0
    tbl["rows"]    = n
    tbl = tbl.sort_values("null_%", ascending=False)
    print(f"\n===== {label}  (rows={n}) =====")
    print(tbl)
    return tbl

# ------------------------------------------------------------------
# 1) Whole-dataset null analysis per kept column
# ------------------------------------------------------------------
null_table(df, COLS, "ALL DATA — null analysis per kept column")

# ------------------------------------------------------------------
# 2) Per-station null analysis (all dates)
# ------------------------------------------------------------------
print("\n\n########## PER-STATION NULL % (ALL DATES) ##########")
per_station_all = {}
for st in stations:
    sub = df[df["station_name"] == st]
    per_station_all[st] = (sub[COLS].isna().mean() * 100).round(1)
per_station_all = pd.DataFrame(per_station_all)
print(per_station_all)

# ------------------------------------------------------------------
# 3) Per-station null analysis (AUGUST 2025 only)
# ------------------------------------------------------------------
print("\n\n########## PER-STATION NULL % (AUG 2025) ##########")
aug_df = df[aug]
per_station_aug = {}
rows_aug = {}
for st in stations:
    sub = aug_df[aug_df["station_name"] == st]
    rows_aug[st] = len(sub)
    per_station_aug[st] = (sub[COLS].isna().mean() * 100).round(1)
per_station_aug = pd.DataFrame(per_station_aug)
print(per_station_aug)
print("\nAugust row counts per station:")
print(pd.Series(rows_aug, name="rows_aug"))

# ------------------------------------------------------------------
# 4) Compact raw null-count matrix per station (August 2025)
# ------------------------------------------------------------------
print("\n\n########## RAW NULL COUNTS PER STATION (AUG 2025) ##########")
aug_counts = {}
for st in stations:
    sub = aug_df[aug_df["station_name"] == st]
    aug_counts[st] = sub[COLS].isna().sum()
print(pd.DataFrame(aug_counts).fillna(0).astype(int))

# ------------------------------------------------------------------
# 5) Optional: row-level: timestamps where a given pollutant is null
#    (top offenders, useful for the fill decision)
# ------------------------------------------------------------------
print("\n\n########## AUG 2025 NULLS ONLY — Lodhi Road vs Shadipur ##########")
focus = ["Lodhi Road", "Shadipur"]
focus_present = [s for s in focus if s in stations]
if focus_present:
    sub = aug_df[aug_df["station_name"].isin(focus_present)]
    print(sub.groupby("station_name")[COLS].apply(lambda g: g.isna().sum()).T)


===== ALL DATA — null analysis per kept column  (rows=131400) =====
                       non_null   null  null_%    rows
NH3 (µg/m³)              119092  12308     9.4  131400
PM10 (µg/m³)             124022   7378     5.6  131400
PM2.5 (µg/m³)            125246   6154     4.7  131400
NO2 (µg/m³)              125684   5716     4.4  131400
Ozone (µg/m³)            125759   5641     4.3  131400
CO (mg/m³)               126278   5122     3.9  131400
NO (µg/m³)               126451   4949     3.8  131400
temperature_2m           131400      0     0.0  131400
relative_humidity_2m     131400      0     0.0  131400
wind_speed_10m           131400      0     0.0  131400
wind_direction_10m       131400      0     0.0  131400
rain                     131400      0     0.0  131400
surface_pressure         131400      0     0.0  131400
boundary_layer_height    131400      0     0.0  131400
wind_u                   131400      0     0.0  131400
wind_v                   131400      0     0.0  131

In [ ]:
# # ffill_missing.py
# import pandas as pd

# FILE = r"D:\miniproject\data\bronze\delhi_air_weather.csv"
OUT  = r"D:\miniproject\data\bronze\delhi_air_weather_ffill.csv"

POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)',
              'NH3 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
METEO = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
         'wind_direction_10m', 'rain', 'surface_pressure',
         'boundary_layer_height', 'wind_u', 'wind_v']

COLS = POLLUTANTS + METEO

# df = pd.read_csv(FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# sort by station then time so ffill respects chronology
df = df.sort_values(["station_name", "timestamp"]).reset_index(drop=True)

# --- before ---
before = df[COLS].isna().sum()

# --- forward fill, per station ---
# optional: add limit=N to avoid carrying stale values across long gaps,
# e.g. limit=3 to fill at most 3 consecutive hours
df[COLS] = (
    df.groupby("station_name", group_keys=False)[COLS]
      .ffill()          # -> .ffill(limit=3) if you want to cap gap propagation
)

# --- after ---
after = df[COLS].isna().sum()

report = pd.DataFrame({
    "null_before": before,
    "null_after":  after,
})
report["filled"]      = report["null_before"] - report["null_after"]
report["remaining_%"] = (report["null_after"] / len(df) * 100).round(2)
report = report.sort_values("null_before", ascending=False)

print("===== FFILL REPORT (per column, whole dataset) =====")
print(report)

# per-station remaining nulls for the target columns
print("\n===== REMAINING NULL % PER STATION (after ffill) =====")
per_station = {}
for st in df["station_name"].dropna().unique():
    sub = df[df["station_name"] == st]
    per_station[st] = (sub[COLS].isna().mean() * 100).round(1)
print(pd.DataFrame(per_station))

df.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}")

===== FFILL REPORT (per column, whole dataset) =====
                       null_before  null_after  filled  remaining_%
NH3 (µg/m³)                  12308        8760    3548         6.67
PM10 (µg/m³)                  7378           0    7378         0.00
PM2.5 (µg/m³)                 6154           0    6154         0.00
NO2 (µg/m³)                   5716           4    5712         0.00
Ozone (µg/m³)                 5641           8    5633         0.01
CO (mg/m³)                    5122           0    5122         0.00
NO (µg/m³)                    4949           0    4949         0.00
temperature_2m                   0           0       0         0.00
relative_humidity_2m             0           0       0         0.00
wind_speed_10m                   0           0       0         0.00
wind_direction_10m               0           0       0         0.00
rain                             0           0       0         0.00
surface_pressure                 0           0       0         

In [32]:
import pandas as pd
OUT  = r"D:\miniproject\data\bronze\delhi_air_weather_ffill.csv"
df = pd.read_csv(OUT)
df.drop(columns=["pm25_filled_from_shadipur", "NH3 (µg/m³)", "precipitation"], inplace=True, errors="ignore")
# df.to_csv(OUT, index=False)

In [33]:
print(df.columns)

Index(['timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
       'NO2 (µg/m³)', 'NOx (ppb)', 'SO2 (µg/m³)', 'CO (mg/m³)',
       'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)',
       'O Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)',
       'AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)',
       'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)', 'station_name', 'latitude_x',
       'longitude_x', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_direction_10m', 'rain', 'surface_pressure',
       'boundary_layer_height', 'wind_u', 'wind_v', 'latitude_y',
       'longitude_y'],
      dtype='object')


In [34]:
Kept_pollutants = ['PM2.5 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
Kept_meteo = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'rain', 'surface_pressure', 'boundary_layer_height', 'wind_u', 'wind_v']
Kept_others = ['station_name', 'timestamp', 'latitude_x', 'longitude_x']
df = df[Kept_pollutants + Kept_meteo + Kept_others]

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131400 entries, 0 to 131399
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   PM2.5 (µg/m³)          131400 non-null  float64
 1   NO (µg/m³)             131400 non-null  float64
 2   NO2 (µg/m³)            131396 non-null  float64
 3   CO (mg/m³)             131400 non-null  float64
 4   Ozone (µg/m³)          131392 non-null  float64
 5   temperature_2m         131400 non-null  float64
 6   relative_humidity_2m   131400 non-null  int64  
 7   wind_speed_10m         131400 non-null  float64
 8   rain                   131400 non-null  float64
 9   surface_pressure       131400 non-null  float64
 10  boundary_layer_height  131400 non-null  float64
 11  wind_u                 131400 non-null  float64
 12  wind_v                 131400 non-null  float64
 13  station_name           131400 non-null  object 
 14  timestamp              131400 non-nu

In [36]:
RENAME = {
    'timestamp'           : 'timestamp',
    'PM2.5 (µg/m³)'       : 'pm25',
    'NO (µg/m³)'          : 'no',
    'NO2 (µg/m³)'         : 'no2',
    'CO (mg/m³)'          : 'co',
    'Ozone (µg/m³)'       : 'o3',
    'temperature_2m'      : 'temp',
    'relative_humidity_2m': 'rh',
    'wind_speed_10m'      : 'ws',
    'rain'                : 'rain',
    'surface_pressure'    : 'pressure',
    'boundary_layer_height': 'blh',
    'wind_u'              : 'u',
    'wind_v'              : 'v',
    'station_name'        : 'station',
    'latitude_x'          : 'lat',
    'longitude_x'         : 'lon',
}

In [37]:
df = df.rename(columns=RENAME)
df.info()
df['timestamp'] = pd.to_datetime(df['timestamp'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131400 entries, 0 to 131399
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   pm25       131400 non-null  float64
 1   no         131400 non-null  float64
 2   no2        131396 non-null  float64
 3   co         131400 non-null  float64
 4   o3         131392 non-null  float64
 5   temp       131400 non-null  float64
 6   rh         131400 non-null  int64  
 7   ws         131400 non-null  float64
 8   rain       131400 non-null  float64
 9   pressure   131400 non-null  float64
 10  blh        131400 non-null  float64
 11  u          131400 non-null  float64
 12  v          131400 non-null  float64
 13  station    131400 non-null  object 
 14  timestamp  131400 non-null  object 
 15  lat        131400 non-null  float64
 16  lon        131400 non-null  float64
dtypes: float64(14), int64(1), object(2)
memory usage: 17.0+ MB


In [28]:
df = df.set_index('timestamp').sort_index()

In [38]:
df.head(20)

,pm25,no,no2,co,o3,temp,rh,ws,rain,pressure,blh,u,v,station,timestamp,lat,lon
0,199.0,20.33,33.20,1.58,8.77,10.3,94,1.77,0.0,994.8,135.0,0.663054,-1.641115,Alipur,2025-01-01 00:00:00,28.8153,77.153
1,190.0,15.38,31.18,1.43,9.10,9.6,94,1.70,0.0,994.3,140.0,0.798102,-1.501011,Alipur,2025-01-01 01:00:00,28.8153,77.153
2,168.0,11.50,29.80,1.43,9.00,9.6,94,1.74,0.0,994.0,110.0,1.095017,-1.352234,Alipur,2025-01-01 02:00:00,28.8153,77.153
3,141.0,6.62,30.70,1.00,11.92,10.2,93,0.95,0.0,994.2,45.0,0.903504,-0.293566,Alipur,2025-01-01 03:00:00,28.8153,77.153
4,110.0,3.25,29.00,0.93,14.28,9.9,92,1.11,0.0,993.7,45.0,1.099198,-0.154482,Alipur,2025-01-01 04:00:00,28.8153,77.153
5,111.0,2.92,31.90,0.82,21.48,10.2,91,1.26,0.0,993.7,60.0,1.250608,-0.153555,Alipur,2025-01-01 05:00:00,28.8153,77.153
6,111.0,2.45,27.80,0.71,37.02,9.9,92,1.81,0.0,993.5,70.0,1.803112,-0.157752,Alipur,2025-01-01 06:00:00,28.8153,77.153
7,99.0,2.30,26.62,0.68,39.80,10.0,92,2.43,0.0,994.1,105.0,2.406351,-0.338191,Alipur,2025-01-01 07:00:00,28.8153,77.153
8,105.0,2.02,27.42,0.79,39.35,10.2,88,2.71,0.0,994.6,220.0,2.650780,-0.563441,Alipur,2025-01-01 08:00:00,28.8153,77.153
9,113.0,2.75,24.58,0.78,42.60,10.0,83,3.29,0.0,995.3,315.0,3.091589,-1.125246,Alipur,2025-01-01 09:00:00,28.8153,77.153


In [39]:
# Handling remaining nulls

# df = df.sort_values(["station", "timestamp"]).reset_index(drop=True)

num_cols = ['pm25','no','no2','co','o3','temp','rh','ws','rain','pressure','blh','u','v']

before = df[num_cols].isna().sum()

# forward fill within each station, respecting time order
df[num_cols] = (
    df.groupby("station", group_keys=False)[num_cols]
      .ffill()
)

after = df[num_cols].isna().sum()

report = pd.DataFrame({
    "null_before": before,
    "null_after":  after,
    "filled":      before - after,
    "remaining_%": (after / len(df) * 100).round(2),
})
print(report.to_string())


          null_before  null_after  filled  remaining_%
pm25                0           0       0         0.00
no                  0           0       0         0.00
no2                 4           4       0         0.00
co                  0           0       0         0.00
o3                  8           8       0         0.01
temp                0           0       0         0.00
rh                  0           0       0         0.00
ws                  0           0       0         0.00
rain                0           0       0         0.00
pressure            0           0       0         0.00
blh                 0           0       0         0.00
u                   0           0       0         0.00
v                   0           0       0         0.00


In [40]:
df.head(20)

,pm25,no,no2,co,o3,temp,rh,ws,rain,pressure,blh,u,v,station,timestamp,lat,lon
0,199.0,20.33,33.20,1.58,8.77,10.3,94,1.77,0.0,994.8,135.0,0.663054,-1.641115,Alipur,2025-01-01 00:00:00,28.8153,77.153
1,190.0,15.38,31.18,1.43,9.10,9.6,94,1.70,0.0,994.3,140.0,0.798102,-1.501011,Alipur,2025-01-01 01:00:00,28.8153,77.153
2,168.0,11.50,29.80,1.43,9.00,9.6,94,1.74,0.0,994.0,110.0,1.095017,-1.352234,Alipur,2025-01-01 02:00:00,28.8153,77.153
3,141.0,6.62,30.70,1.00,11.92,10.2,93,0.95,0.0,994.2,45.0,0.903504,-0.293566,Alipur,2025-01-01 03:00:00,28.8153,77.153
4,110.0,3.25,29.00,0.93,14.28,9.9,92,1.11,0.0,993.7,45.0,1.099198,-0.154482,Alipur,2025-01-01 04:00:00,28.8153,77.153
5,111.0,2.92,31.90,0.82,21.48,10.2,91,1.26,0.0,993.7,60.0,1.250608,-0.153555,Alipur,2025-01-01 05:00:00,28.8153,77.153
6,111.0,2.45,27.80,0.71,37.02,9.9,92,1.81,0.0,993.5,70.0,1.803112,-0.157752,Alipur,2025-01-01 06:00:00,28.8153,77.153
7,99.0,2.30,26.62,0.68,39.80,10.0,92,2.43,0.0,994.1,105.0,2.406351,-0.338191,Alipur,2025-01-01 07:00:00,28.8153,77.153
8,105.0,2.02,27.42,0.79,39.35,10.2,88,2.71,0.0,994.6,220.0,2.650780,-0.563441,Alipur,2025-01-01 08:00:00,28.8153,77.153
9,113.0,2.75,24.58,0.78,42.60,10.0,83,3.29,0.0,995.3,315.0,3.091589,-1.125246,Alipur,2025-01-01 09:00:00,28.8153,77.153


In [42]:
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='mean')
df[num_cols] = imp.fit_transform(df[num_cols])
after = df[num_cols].isna().sum()

In [43]:
after

pm25        0
no          0
no2         0
co          0
o3          0
temp        0
rh          0
ws          0
rain        0
pressure    0
blh         0
u           0
v           0
dtype: int64

In [44]:
df.head(20)

,pm25,no,no2,co,o3,temp,rh,ws,rain,pressure,blh,u,v,station,timestamp,lat,lon
0,199.0,20.33,33.20,1.58,8.77,10.3,94.0,1.77,0.0,994.8,135.0,0.663054,-1.641115,Alipur,2025-01-01 00:00:00,28.8153,77.153
1,190.0,15.38,31.18,1.43,9.10,9.6,94.0,1.70,0.0,994.3,140.0,0.798102,-1.501011,Alipur,2025-01-01 01:00:00,28.8153,77.153
2,168.0,11.50,29.80,1.43,9.00,9.6,94.0,1.74,0.0,994.0,110.0,1.095017,-1.352234,Alipur,2025-01-01 02:00:00,28.8153,77.153
3,141.0,6.62,30.70,1.00,11.92,10.2,93.0,0.95,0.0,994.2,45.0,0.903504,-0.293566,Alipur,2025-01-01 03:00:00,28.8153,77.153
4,110.0,3.25,29.00,0.93,14.28,9.9,92.0,1.11,0.0,993.7,45.0,1.099198,-0.154482,Alipur,2025-01-01 04:00:00,28.8153,77.153
5,111.0,2.92,31.90,0.82,21.48,10.2,91.0,1.26,0.0,993.7,60.0,1.250608,-0.153555,Alipur,2025-01-01 05:00:00,28.8153,77.153
6,111.0,2.45,27.80,0.71,37.02,9.9,92.0,1.81,0.0,993.5,70.0,1.803112,-0.157752,Alipur,2025-01-01 06:00:00,28.8153,77.153
7,99.0,2.30,26.62,0.68,39.80,10.0,92.0,2.43,0.0,994.1,105.0,2.406351,-0.338191,Alipur,2025-01-01 07:00:00,28.8153,77.153
8,105.0,2.02,27.42,0.79,39.35,10.2,88.0,2.71,0.0,994.6,220.0,2.650780,-0.563441,Alipur,2025-01-01 08:00:00,28.8153,77.153
9,113.0,2.75,24.58,0.78,42.60,10.0,83.0,3.29,0.0,995.3,315.0,3.091589,-1.125246,Alipur,2025-01-01 09:00:00,28.8153,77.153


In [45]:
OUT1 = r"D:\miniproject\data\silver\delhi_air_weather.csv"
df.to_csv(OUT1, index=True)

In [23]:
df.drop(columns = ["pm10"], inplace=True, errors="ignore")

In [25]:
df.head()

,pm25,no,no2,co,o3,temp,rh,ws,rain,pressure,blh,u,v,station
timestamp,,,,,,,,,,,,,,
2025-01-01 00:00:00,199.00,20.33,33.20,1.58,8.77,10.3,94,1.77,0.0,994.8,135.0,0.663054,-1.641115,Alipur
2025-01-01 00:00:00,200.41,137.17,NaN,1.51,2.75,10.0,94,1.81,0.0,994.7,75.0,0.189197,-1.800085,Lodhi Road
2025-01-01 00:00:00,282.00,155.83,48.63,1.85,7.17,9.4,95,1.58,0.0,994.9,90.0,0.301478,-1.550971,Mundka
2025-01-01 00:00:00,101.00,12.80,40.53,0.90,8.53,10.3,93,1.81,0.0,994.9,60.0,-0.157752,-1.803112,Najafgarh
2025-01-01 00:00:00,203.20,21.11,75.32,0.87,21.29,10.2,94,1.77,0.0,993.8,135.0,0.663054,-1.641115,Shadipur


In [26]:
df.to_csv(OUT1, index=True)

In [3]:
import numpy as np
import pandas as pd

CSV_FILE = "D:/miniproject/data/silver/delhi_air_weather.csv"
df = pd.read_csv(CSV_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"])
unique_times = np.sort(df["timestamp"].dropna().unique())

In [4]:
print(f"Total unique timestamps: {len(unique_times)}")

Total unique timestamps: 8760


In [5]:
split_idx = int(len(unique_times) * 0.8)
cutoff = unique_times[split_idx]

In [6]:
print(f"end=train cutoff: {cutoff}")

end=train cutoff: 2025-10-20T00:00:00.000000000


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# Convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort by timestamp
df = df.sort_values("timestamp").reset_index(drop=True)

# Split unique timestamps, NOT individual rows
unique_times = np.sort(df["timestamp"].dropna().unique())

# Five expanding-window folds
tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    tscv.split(unique_times), start=1
):
    train_times = unique_times[train_idx]
    val_times = unique_times[val_idx]

    train = df[df["timestamp"].isin(train_times)].copy()
    val = df[df["timestamp"].isin(val_times)].copy()

    print(f"\nFold {fold}")
    print("Train period:",
          train["timestamp"].min(), "to",
          train["timestamp"].max())
    print("Validation period:",
          val["timestamp"].min(), "to",
          val["timestamp"].max())
    print("Train rows:", len(train))
    print("Validation rows:", len(val))
    print("Training stations:", train["station"].nunique())
    print("Validation stations:", val["station"].nunique())


Fold 1
Train period: 2025-01-01 00:00:00 to 2025-03-02 19:00:00
Validation period: 2025-03-02 20:00:00 to 2025-05-02 15:00:00
Train rows: 21900
Validation rows: 21900
Training stations: 15
Validation stations: 15

Fold 2
Train period: 2025-01-01 00:00:00 to 2025-05-02 15:00:00
Validation period: 2025-05-02 16:00:00 to 2025-07-02 11:00:00
Train rows: 43800
Validation rows: 21900
Training stations: 15
Validation stations: 15

Fold 3
Train period: 2025-01-01 00:00:00 to 2025-07-02 11:00:00
Validation period: 2025-07-02 12:00:00 to 2025-09-01 07:00:00
Train rows: 65700
Validation rows: 21900
Training stations: 15
Validation stations: 15

Fold 4
Train period: 2025-01-01 00:00:00 to 2025-09-01 07:00:00
Validation period: 2025-09-01 08:00:00 to 2025-11-01 03:00:00
Train rows: 87600
Validation rows: 21900
Training stations: 15
Validation stations: 15

Fold 5
Train period: 2025-01-01 00:00:00 to 2025-11-01 03:00:00
Validation period: 2025-11-01 04:00:00 to 2025-12-31 23:00:00
Train rows: 10950

In [8]:
train_stations = set(train["station"].unique())
val_stations = set(val["station"].unique())

unseen_stations = val_stations - train_stations

print("Stations unseen during training:", unseen_stations)

Stations unseen during training: set()
